# 第 1 周 · 第 1 天练习（Delivery / Project Management 助手）

## 练习目标（理念）

封装一次 **OpenAI Chat Completions** 调用：用 **system prompt** 固定「交付管理 / 项目管理」专家角色，用 **user prompt** 传入具体业务场景，得到可执行的交付物（执行摘要、风险、下一步等）。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 环境变量里的 API Key | `os.getenv("OPENAI_API_KEY")` |
| `messages`（system / user） | system 定领域与语气；user 放会议/项目上下文 |
| Chat Completions | `client.chat.completions.create(...)` |

## 怎么跑

1. 设置环境变量 `OPENAI_API_KEY`
2. 运行本笔记本唯一代码格（或当脚本执行）
3. 可改 `__main__` 里的 `message` 换成你自己的项目跟进上下文

> 说明：发给模型的 system/user 正文保持**西班牙语原文**（影响回答语言与风格）；旁注为中文教学注释。


In [ ]:
# ========== 导入：环境变量 + OpenAI 客户端 ==========

# 导入标准库 os：从环境读取 OPENAI_API_KEY
import os
# 从 openai 导入 OpenAI：调用云端 Chat Completions API
from openai import OpenAI


def call_openai(message: str) -> str:
    """用 system + user 两段 messages 调用 OpenAI 前沿模型，返回助手回复文本。"""

    # 从环境变量读取 API Key（不要把密钥写进代码）
    api_key = os.getenv("OPENAI_API_KEY")

    # 没有密钥就立刻失败，避免默默带着空 key 去请求
    if not api_key:
        raise ValueError(
            "No se ha encontrado la variable de entorno OPENAI_API_KEY. "
            "Configúrala antes de ejecutar el script."
        )

    # 用显式 api_key 创建客户端
    client = OpenAI(api_key=api_key)

    # system prompt 保留原文（西班牙语）：定义领域角色、交付物类型与语气；翻译会改变模型行为
    system_prompt = """
Eres un asistente experto en Delivery Management y Project Management.
Tu tarea es ayudar a transformar información operativa en entregables útiles:
resúmenes ejecutivos, planes de acción, riesgos, actas, análisis de desviaciones,
preparación de comités, reporting de proyecto y comunicación con cliente.

Usa un tono profesional, claro, directo y orientado a la acción.
Evita florituras. Prioriza estructura, precisión y utilidad práctica.
Cuando falten datos, identifica supuestos razonables y riesgos.
"""

    # messages：system 定角色；user 放本次具体业务请求（message 参数）
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": message}
    ]

    # 调用 Chat Completions；模型 id 保持原样（影响计费与能力）
    response = client.chat.completions.create(
        model="gpt-5.5",
        messages=messages
    )

    # 取出第一条 choice 的助手正文
    return response.choices[0].message.content


# ========== 脚本入口：示例项目跟进会 → 要执行摘要 / 风险 / 下一步 ==========

if __name__ == "__main__":
    # user 消息保留西班牙语原文：这是发给模型的业务输入，不要翻译
    message = """
Tengo una reunión de seguimiento de un proyecto GIS.
El cliente está preocupado porque las pruebas UAT se retrasan por problemas de acceso,
hay dependencias con infraestructura y todavía no está cerrado el plan de pruebas.
Necesito un resumen ejecutivo, riesgos y próximos pasos.
"""

    # 调用封装函数并打印模型回复
    result = call_openai(message)
    print(result)
